In [8]:
# Cargar paquetes necesarios
# Instalar paquetes si es necesario
using Pkg
packages = ["Random", "Statistics", "DataFrames", "GLM", "StatsBase", "Distributions", "HypothesisTests", "MLJ", "MLJLinearModels"]

for pkg in packages
    try
        eval(Meta.parse("using $pkg"))
    catch
        Pkg.add(pkg)
        eval(Meta.parse("using $pkg"))
    end
end

println("¡Todas las librerías están listas!")

¡Todas las librerías están listas!


In [9]:
# SIMULACIÓN DE DATOS
Random.seed!(42)
n = 1000

# Generar variables
X1 = randn(n)
X3 = randn(n)
X2 = rand(Bernoulli(0.5), n)
X4 = rand(Bernoulli(0.5), n)
D = rand(Bernoulli(0.5), n)
epsilon = randn(n)
# 3. VARIABLE OUTCOME Y según la fórmula del lab 3:
# Variable outcome
Y = 2 .* D + 0.5 .* X1 - 0.3 .* X2 + 0.2 .* X3 + epsilon
# Note que es el proceso generador de datos. Y no depende de X4 lo que nos
#lleva a intuir que X4 no es una variable relevante.
# Crear DataFrame
df = DataFrame(Y=Y, D=D, X1=X1, X2=X2, X3=X3, X4=X4)

Row,Y,D,X1,X2,X3,X4
,Float64,Bool,Float64,Bool,Float64,Bool
1,-0.990813,false,-0.363357,true,-0.01988,true
2,1.54639,true,0.251737,false,0.218246,true
3,0.342411,false,-0.314988,true,1.52856,false
4,1.4503,true,-0.311252,true,-0.764316,true
5,-1.63479,false,0.816307,false,-2.02765,false
6,2.52277,true,0.476738,false,1.02999,false
7,0.0861608,false,-0.859555,false,0.981408,false
8,1.64711,true,-1.46929,false,-0.513191,true
9,1.6688,true,-2.11433,false,0.613236,true


In [10]:
println("Primeras 5 filas del DataFrame:")
println(first(df, 5))
println("Dimensión: $(size(df))")

# Balance check
tratamiento = df[df.D .== 1, :]
control = df[df.D .== 0, :]

println("\nBALANCE CHECK:")
for var in [:X1, :X2, :X3, :X4]
    test_result = UnequalVarianceTTest(tratamiento[!, var], control[!, var])
    println("$var - Media T: $(round(mean(tratamiento[!, var]), digits=3)), Media C: $(round(mean(control[!, var]), digits=3)), p-valor: $(round(pvalue(test_result), digits=3))")
end

Primeras 5 filas del DataFrame:
5×6 DataFrame
 Row │ Y          D      X1         X2     X3         X4    
     │ Float64    Bool   Float64    Bool   Float64    Bool  
─────┼──────────────────────────────────────────────────────
   1 │ -0.990813  false  -0.363357   true  -0.01988    true
   2 │  1.54639    true   0.251737  false   0.218246   true
   3 │  0.342411  false  -0.314988   true   1.52856   false
   4 │  1.4503     true  -0.311252   true  -0.764316   true
   5 │ -1.63479   false   0.816307  false  -2.02765   false
Dimensión: (1000, 6)

BALANCE CHECK:
X1 - Media T: -0.043, Media C: -0.077, p-valor: 0.588
X2 - Media T: 0.46, Media C: 0.508, p-valor: 0.126
X3 - Media T: 0.057, Media C: -0.018, p-valor: 0.242
X4 - Media T: 0.52, Media C: 0.524, p-valor: 0.913


In [11]:
# REGRESIONES
println("\nREGRESIONES:")

# Regresión simple
modelo_simple = lm(@formula(Y ~ D), df)
ate_simple = coef(modelo_simple)[2]
se_simple = stderror(modelo_simple)[2]
println("Simple - ATE: $(round(ate_simple, digits=3)), SE: $(round(se_simple, digits=3))")

# Regresión completa
modelo_completo = lm(@formula(Y ~ D + X1 + X2 + X3 + X4), df)
ate_completo = coef(modelo_completo)[2]
se_completo = stderror(modelo_completo)[2]
println("Completa - ATE: $(round(ate_completo, digits=3)), SE: $(round(se_completo, digits=3))")

println("Diferencia ATE: $(round(ate_completo - ate_simple, digits=3))")


REGRESIONES:
Simple - ATE: 2.018, SE: 0.071
Completa - ATE: 1.974, SE: 0.063
Diferencia ATE: -0.044


In [12]:
# LASSO Y SELECCIÓN DE VARIABLES
println("\nLASSO:")

# Lambda teórico de Chernozhukov
function chernozhukov_lambda(n, p)
    c = 1.1
    alpha = 0.05
    return c * sqrt(n) * quantile(Normal(), 1 - alpha / (2 * p))
end

# Preparar datos
X_cov = Matrix(df[:, [:X1, :X2, :X3, :X4]])
y = df.Y

# Estandarizar X1 y X3
X_prep = copy(X_cov)
X_prep[:, 1] = (X_prep[:, 1] .- mean(X_prep[:, 1])) ./ std(X_prep[:, 1])
X_prep[:, 3] = (X_prep[:, 3] .- mean(X_prep[:, 3])) ./ std(X_prep[:, 3])

p = 4
lambda_teorico = chernozhukov_lambda(n, p)
println("Lambda teórico: $(round(lambda_teorico, digits=6))")


LASSO:
Lambda teórico: 86.88282


In [13]:
# LASSO teórico usando GLMNet
# GLMNet espera lambda sin dividir por n
lasso_teo = glmnet(X_prep, y, lambda=[lambda_teorico], alpha=1.0)
coefs_teo = lasso_teo.betas[:, 1]  # Coeficientes para el primer (y único) lambda

# LASSO con validación cruzada usando GLMNet
cv_result = glmnetcv(X_prep, y, alpha=1.0, nfolds=5)
best_lambda = cv_result.lambda[argmin(cv_result.meanloss)]
best_coefs = cv_result.path.betas[:, argmin(cv_result.meanloss)]

println("Lambda CV: $(round(best_lambda, digits=6))")

# Resultados LASSO
variables = ["X1", "X2", "X3", "X4"]
umbral = 1e-6

println("\nCoeficientes LASSO teórico:")
for i in 1:4
    println("$(variables[i]): $(round(coefs_teo[i], digits=6))")
end
vars_teo = variables[abs.(coefs_teo) .> umbral]
println("Variables seleccionadas teórico: $vars_teo")

println("\nCoeficientes LASSO CV:")
for i in 1:4
    println("$(variables[i]): $(round(best_coefs[i], digits=6))")
end
vars_cv = variables[abs.(best_coefs) .> umbral]
println("Variables seleccionadas CV: $vars_cv")

Lambda CV: 0.002255

Coeficientes LASSO teórico:
X1: 0.0
X2: 0.0
X3: 0.0
X4: 0.0
Variables seleccionadas teórico: String[]

Coeficientes LASSO CV:
X1: 0.496829
X2: -0.351129
X3: 0.229494
X4: -0.076531
Variables seleccionadas CV: ["X1", "X2", "X3", "X4"]


In [14]:
# Re-estimación con variables seleccionadas
function estimar_ate(vars_seleccionadas)
    if isempty(vars_seleccionadas)
        modelo = lm(@formula(Y ~ D), df)
    else
        formula_str = "Y ~ D + " * join(vars_seleccionadas, " + ")
        formula = eval(Meta.parse("@formula($formula_str)"))
        modelo = lm(formula, df)
    end
    return coef(modelo)[2], stderror(modelo)[2]
end

ate_teo, se_teo = estimar_ate(vars_teo)
ate_cv, se_cv = estimar_ate(vars_cv)

println("\nRESULTADOS FINALES:")
println("ATE verdadero: 2.000")
println("ATE teórico: $(round(ate_teo, digits=3))")
println("ATE CV: $(round(ate_cv, digits=3))")
println("ATE completo: $(round(ate_completo, digits=3))")
println("ATE simple: $(round(ate_simple, digits=3))")


RESULTADOS FINALES:
ATE verdadero: 2.000
ATE teórico: 2.018
ATE CV: 1.974
ATE completo: 1.974
ATE simple: 2.018
